In [1]:
import pandas as pd
import numpy as np

In [2]:
df= pd.read_csv("../raw_data/amazon_india_2020.csv")

In [3]:
df.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2020_00000001,2020-01-19,CUST_2020_00001031,PROD_000402,Xiaomi Poco F1 128GB Blue,Electronics,Smartphones,Xiaomi,36202.67,7.11,...,False,NaN,4.5,Cancelled,1,2020,1,0.19,False,3.4
1,TXN_2020_00000002,2020-01-19,CUST_2018_00013266,PROD_000486,Apple iPhone 11 128GB Blue,Electronics,Smartphones,Apple,165163.92,0.00,...,False,NaN,NaN,Returned,1,2020,1,0.19,True,4.3
2,TXN_2020_00000003,2020-01-17,CUST_2020_00001673,PROD_000278,Xiaomi Redmi 4A 16GB White,Electronics,Smartphones,Xiaomi,32906.94,0.00,...,False,NaN,5.0,Delivered,1,2020,1,0.21,True,4.1
3,TXN_2020_00000004,2020-01-25,CUST_2020_00026212,PROD_000245,Samsung Galaxy S8+ 32GB Black,Electronics,Smartphones,Samsung,136675.37,45.87,...,True,Republic Day Sale,4.5,Delivered,1,2020,1,0.23,True,3.8
4,TXN_2020_00000005,2020-01-01,CUST_2015_00009980,PROD_000765,Vivo V20 64GB Black,Electronics,Smartphones,Vivo,30041.81,0.00,...,False,NaN,4.0,Delivered,1,2020,1,0.23,False,3.9


In [4]:
df["delivery_charges"].isna().sum(), len(df)

(np.int64(11518), 143715)

In [5]:
df["delivery_charges"].describe()

count    132197.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
Name: delivery_charges, dtype: float64

In [6]:
df.drop(columns=["delivery_charges"], inplace=True)

In [7]:
df.columns

Index(['transaction_id', 'order_date', 'customer_id', 'product_id',
       'product_name', 'category', 'subcategory', 'brand',
       'original_price_inr', 'discount_percent', 'discounted_price_inr',
       'quantity', 'subtotal_inr', 'final_amount_inr', 'customer_city',
       'customer_state', 'customer_tier', 'customer_spending_tier',
       'customer_age_group', 'payment_method', 'delivery_days',
       'delivery_type', 'is_prime_member', 'is_festival_sale', 'festival_name',
       'customer_rating', 'return_status', 'order_month', 'order_year',
       'order_quarter', 'product_weight_kg', 'is_prime_eligible',
       'product_rating'],
      dtype='object')

Question 1
Your dataset contains order_date in multiple formats: 'DD/MM/YYYY', 'DD-MM-YY', 'YYYY-MM-DD', and some invalid entries like '32/13/2020'. Clean and standardize all dates to 'YYYY-MM-DD' format, handling invalid dates appropriately.


In [8]:
df["order_date"].head(20)

0     2020-01-19
1     2020-01-19
2     2020-01-17
3     2020-01-25
4     2020-01-01
5     2020-01-13
6     2020-01-18
7     2020-01-04
8     2020-01-26
9     22-01-2020
10    2020-01-11
11    2020-01-16
12    2020-01-26
13    2020-01-22
14    2020-01-15
15    2020-01-15
16    2020-01-05
17    2020-01-14
18    2020-01-06
19    2020-01-08
Name: order_date, dtype: object

In [9]:
df["order_date"] = (
    df["order_date"]
    .astype(str)
    .str.replace("/", "-", regex=False)
    .str.replace(" ", "", regex=False)
)


parts= df["order_date"].str.split("-", expand=True)
year_last= parts[2].str. len()==4
df.loc[year_last,"order_date"]=(parts[2] + "-" + parts[0] + "-" + parts[1])
parts=df["order_date"].str.split("-", expand=True)
mask= parts[1].astype(int)>12
df.loc[mask, "order_date"]= (parts[0]+"-"+parts[2]+"-"+parts[1])
df["order_date"]= pd.to_datetime(df["order_date"], errors= "coerce")

In [10]:
df["order_date"].min(), df["order_date"].max()


(Timestamp('2020-01-01 00:00:00'), Timestamp('2020-12-31 00:00:00'))

In [11]:
df["order_date"].isna().sum()

np.int64(0)

Question 2
The original_price_inr column contains mixed data types: numeric values, text with '₹' symbols, comma separators ('₹1,25,000'), and some entries like 'Price on Request'. Clean this column to contain only numeric values in Indian Rupees. 


In [12]:
df["original_price_inr"].head(20)

0        36202.67
1       165163.92
2        32906.94
3       136675.37
4        30041.81
5        35361.65
6      ₹57,597.55
7       Rs 53,908
8        53640.12
9        38891.76
10      8103463.0
11     ₹35,279.33
12      117219.84
13       95257.81
14       44554.25
15       31895.68
16       29935.32
17    ₹211,304.52
18      156946.28
19       49163.92
Name: original_price_inr, dtype: object

In [13]:
df["original_price_inr"]= df["original_price_inr"].astype(str)
df["original_price_inr"] = df["original_price_inr"].str.replace(" ","",regex=False)
df["original_price_inr"]= df["original_price_inr"].str.replace(",", "",regex=False)
df["original_price_inr"] = df["original_price_inr"].str.replace("₹", "", regex=False)
df["original_price_inr"]= pd.to_numeric(df["original_price_inr"], errors="coerce")

In [14]:
df["original_price_inr"].unique()[:20]

array([  36202.67,  165163.92,   32906.94,  136675.37,   30041.81,
         35361.65,   57597.55,        nan,   53640.12,   38891.76,
       8103463.  ,   35279.33,  117219.84,   95257.81,   44554.25,
         31895.68,   29935.32,  211304.52,  156946.28,   49163.92])

In [15]:
mask = df["original_price_inr"].isna()

df.loc[mask, "original_price_inr"] = np.where(
    df.loc[mask, "discount_percent"] == 0,
    
    # Case 1: no discount
    df.loc[mask, "discounted_price_inr"],
    
    # Case 2: discount present
    df.loc[mask, "discounted_price_inr"] / (1 - df.loc[mask, "discount_percent"] / 100)
)

In [16]:
df["original_price_inr"].isna().sum()

np.int64(0)

In [17]:
df["original_price_inr"].dtypes

dtype('float64')

Question 3
Customer ratings appear in various formats: '5.0', '4 stars', '3/5', '2.5/5.0', and some missing values. Standardize all ratings to numeric scale 1.0-5.0, handling inconsistent formats and missing values strategically.


In [18]:
df["customer_rating"]=df["customer_rating"].astype(str)
df["customer_rating"]= df["customer_rating"].str.replace("stars","",regex=False)
df["customer_rating"]= df["customer_rating"].str.split("/").str[0]
df["customer_rating"]= pd.to_numeric(df["customer_rating"],errors="coerce")


In [19]:
df["customer_rating"].describe()

count    100090.000000
mean          4.312778
std           0.570392
min           3.000000
25%           4.000000
50%           4.500000
75%           5.000000
max           5.000000
Name: customer_rating, dtype: float64

In [20]:
df["customer_rating"].value_counts().head(10)

customer_rating
4.5    33251
5.0    25642
4.0    25114
3.5    10243
3.0     5840
Name: count, dtype: int64

In [21]:
df["customer_rating"].isna().sum()

np.int64(43625)

In [22]:
df["customer_rating"].value_counts(dropna=False)

customer_rating
NaN    43625
4.5    33251
5.0    25642
4.0    25114
3.5    10243
3.0     5840
Name: count, dtype: int64

Question 4
The customer_city column has inconsistent naming: 'Bangalore/Bengaluru', 'Mumbai/Bombay', 'Delhi/New Delhi', along with spelling errors and case variations. Standardize all city names and handle geographical variations.


In [23]:
df["customer_city"]= df["customer_city"].str.strip().str.lower()

In [24]:
df["customer_city"].unique()

array(['delhi', 'ahmedabad', 'chennai', 'bhubaneswar', 'bangalore',
       'patna', 'mumbai', 'coimbatore', 'lucknow', 'kolkata', 'indore',
       'hyderabad', 'vadodara', 'pune', 'visakhapatnam', 'jaipur',
       'bareilly', 'kanpur', 'ludhiana', 'meerut', 'saharanpur', 'nagpur',
       'surat', 'chandigarh', 'gorakhpur', 'kochi', 'varanasi',
       'allahabad', 'moradabad', 'aligarh', 'new delhi', 'banglore',
       'delhi ncr', 'mumba', 'calcutta', 'bengaluru', 'bombay', 'chenai',
       'bengalore', 'madras'], dtype=object)

In [25]:
city_map = {
    "new delhi": "delhi",
    "delhi ncr": "delhi",

    "banglore": "bangalore",
    "bengalore": "bangalore",
    "bengaluru": "bangalore",

    "mumba": "mumbai",
    "bombay": "mumbai",

    "calcutta": "kolkata",

    "chenai": "chennai",
    "madras": "chennai"
}

In [26]:
df["customer_city"] = df["customer_city"].replace(city_map)

In [27]:
df["customer_city"]= df["customer_city"].str.title()

In [28]:
df["customer_city"].value_counts().head()

customer_city
Mumbai       18734
Delhi        16266
Bangalore    13096
Chennai      11185
Kolkata       8793
Name: count, dtype: int64

In [29]:
df["customer_city"].unique()

array(['Delhi', 'Ahmedabad', 'Chennai', 'Bhubaneswar', 'Bangalore',
       'Patna', 'Mumbai', 'Coimbatore', 'Lucknow', 'Kolkata', 'Indore',
       'Hyderabad', 'Vadodara', 'Pune', 'Visakhapatnam', 'Jaipur',
       'Bareilly', 'Kanpur', 'Ludhiana', 'Meerut', 'Saharanpur', 'Nagpur',
       'Surat', 'Chandigarh', 'Gorakhpur', 'Kochi', 'Varanasi',
       'Allahabad', 'Moradabad', 'Aligarh'], dtype=object)

Question 5
Boolean columns (is_prime_member, is_prime_eligible, is_festival_sale) contain mixed values: True/False, Yes/No, 1/0, Y/N, and some missing entries. Convert all boolean columns to consistent True/False format.


In [30]:
bool_candidates=[]
bool_values={"true", "false","y","n","yes", "no","0","1"}

for col in df.columns:
    vals= set(df[col].astype(str). str.lower().dropna().unique())
    if vals & bool_values:
        bool_candidates.append(col)
bool_candidates



['quantity',
 'delivery_days',
 'is_prime_member',
 'is_festival_sale',
 'order_month',
 'order_quarter',
 'is_prime_eligible']

In [31]:
boolean_cols = ["is_prime_member", "is_prime_eligible", "is_festival_sale"]

bool_map = {
    "true": True,
    "false": False,
    "yes": True,
    "no": False,
    "y": True,
    "n": False,
    "1": True,
    "0": False
}

for col in boolean_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(bool_map)
    )

In [32]:
cols = ["is_prime_member", "is_prime_eligible", "is_festival_sale"]

for col in cols:
    print(col)
    print(df[col].unique())
    print()

is_prime_member
[ True False]

is_prime_eligible
[False  True]

is_festival_sale
[False  True]



Question 6
Product categories have variations: 'Electronics/Electronic/ELECTRONICS/Electronics & Accessories'. Standardize category names across the dataset and ensure consistent naming conventions.


In [3]:
df["category"].value_counts().head(20)

category
Electronics                  143634
Electronicss                     24
ELECTRONICS                      23
Electronic                       23
Electronics & Accessories        11
Name: count, dtype: int64

In [4]:
category_map = {
    "Electronicss": "Electronics",
    "ELECTRONICS": "Electronics",
    "Electronic": "Electronics",
    "Electronics & Accessories": "Electronics"
}

In [5]:
df["category"] = df["category"].replace(category_map)
df["category"] = df["category"].str.title()

In [6]:
df["category"].value_counts()

category
Electronics    143715
Name: count, dtype: int64

Question 7
The delivery_days column contains negative values, text entries like 'Same Day', '1-2 days', and some unrealistic values like 50 days. Clean this column to contain only valid numeric delivery days.


In [37]:
df["delivery_days"].unique()

array(['1', '4', '5', '3', '6', '15', '2', '0', '-1', '7', '1-2 days',
       'Express', 'Same Day'], dtype=object)

In [38]:
df["delivery_days"] = df["delivery_days"].astype(str).str.strip().str.lower()

In [39]:
df["delivery_days"] = df["delivery_days"].replace({
    "same day": "0",
    "express": "1"
})

In [40]:
df["delivery_days"] = df["delivery_days"].str.extract(r"(-?\d+)")

In [41]:
df["delivery_days"] = pd.to_numeric(df["delivery_days"], errors="coerce")

In [42]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = None

In [43]:
(df["delivery_days"] < 0).sum()

np.int64(0)

In [44]:
df["delivery_days"].unique()

array([ 1.,  4.,  5.,  3.,  6., 15.,  2.,  0., nan,  7.])

In [45]:
df["delivery_days"].isnull().sum()

np.int64(861)

In [46]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = np.nan

df["delivery_days"] = df["delivery_days"].fillna(df["delivery_days"].median())

In [47]:
df["delivery_days"].describe()

count    143715.000000
mean          3.360025
std           1.754949
min           0.000000
25%           2.000000
50%           3.000000
75%           4.000000
max          15.000000
Name: delivery_days, dtype: float64

In [48]:
df["delivery_days"].isna().sum()

np.int64(0)

Question 8
Identify and handle duplicate transactions where the same customer, product, date, and amount appear multiple times. Some duplicates are genuine (bulk orders) while others are data errors. Develop a strategy to distinguish and handle both cases.


In [49]:
duplicate_mask = df.duplicated(
    subset=["customer_id", "product_id", "order_date", "original_price_inr"],
    keep=False
)

duplicates = df[duplicate_mask]

In [50]:
duplicates.shape

(1408, 33)

In [51]:
duplicates.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
341,TXN_2020_00000342,2020-01-06,CUST_2020_00031761,PROD_001614,Apple VivoBook 4GB RAM Silver,Electronics,Laptops,Apple,171440.11,8.47,...,False,NaN,4.0,Cancelled,1,2020,1,1.98,False,3.2
380,TXN_2020_00000381,2020-01-19,CUST_2020_00043247,PROD_000786,Oppo A52 256GB White,Electronics,Smartphones,Oppo,40389.16,22.52,...,False,NaN,NaN,Delivered,1,2020,1,0.24,True,4.4
670,TXN_2020_00000671,2020-01-31,CUST_2020_00046302,PROD_000678,Samsung Galaxy S20+ 64GB Blue,Electronics,Smartphones,Samsung,204873.32,0.00,...,False,NaN,5.0,Delivered,1,2020,1,0.17,False,3.8
781,TXN_2020_00000782,2020-01-24,CUST_2020_00000131,PROD_000189,Xiaomi Redmi 3s 16GB Gold,Electronics,Smartphones,Xiaomi,48821.25,64.39,...,True,Republic Day Sale,NaN,Delivered,1,2020,1,0.18,True,3.7
947,TXN_2020_00000948,2020-01-06,CUST_2020_00044914,PROD_000314,Oppo F5 16GB White,Electronics,Smartphones,Oppo,30856.62,8.40,...,False,NaN,5.0,Delivered,1,2020,1,0.21,True,4.1


In [52]:
df.duplicated().sum()

np.int64(0)

In [53]:
duplicates.sort_values(
    ["customer_id","product_id","order_date"]
).head(10)

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
116596,TXN_2020_00116597,2020-11-20,CUST_2015_00000020,PROD_001826,OnePlus Gaming Headset Premium,Electronics,Audio,OnePlus,26420.31,0.00,...,False,NaN,3.0,Delivered,11,2020,4,0.35,True,4.6
143088,TXN_2020_00116597_DUP,2020-11-20,CUST_2015_00000020,PROD_001826,OnePlus Gaming Headset Premium,Electronics,Audio,OnePlus,26420.31,0.00,...,False,NaN,3.0,Delivered,11,2020,4,0.35,True,4.6
120273,TXN_2020_00120274,2020-11-15,CUST_2015_00000046,PROD_000061,OnePlus OnePlus X 32GB White,Electronics,Smartphones,OnePlus,102173.68,11.89,...,False,NaN,4.0,Delivered,11,2020,4,0.19,True,3.6
143535,TXN_2020_00120274_DUP,2020-11-15,CUST_2015_00000046,PROD_000061,OnePlus OnePlus X 32GB White,Electronics,Smartphones,OnePlus,102173.68,11.89,...,False,NaN,4.0,Delivered,11,2020,4,0.19,True,3.6
53007,TXN_2020_00053008,2020-05-03,CUST_2015_00000122,PROD_000743,Realme Realme X50 Pro 64GB Black,Electronics,Smartphones,Realme,42295.50,62.38,...,True,Summer Sale,4.5,Delivered,5,2020,2,0.23,True,4.5
143477,TXN_2020_00053008_DUP,2020-05-03,CUST_2015_00000122,PROD_000743,Realme Realme X50 Pro 64GB Black,Electronics,Smartphones,Realme,42295.50,62.38,...,True,Summer Sale,4.5,Delivered,5,2020,2,0.23,True,4.5
109457,TXN_2020_00109458,2020-10-28,CUST_2015_00000134,PROD_000040,Samsung Galaxy Note 5 16GB White,Electronics,Smartphones,Samsung,184502.49,57.79,...,True,Diwali Sale,NaN,Delivered,10,2020,4,0.15,True,4.0
143638,TXN_2020_00109458_DUP,2020-10-28,CUST_2015_00000134,PROD_000040,Samsung Galaxy Note 5 16GB White,Electronics,Smartphones,Samsung,184502.49,57.79,...,True,Diwali Sale,NaN,Delivered,10,2020,4,0.15,True,4.0
79960,TXN_2020_00079961,2020-08-15,CUST_2015_00001186,PROD_000646,Apple iPhone 12 Pro 64GB White,Electronics,Smartphones,Apple,223856.56,0.00,...,False,NaN,5.0,Delivered,8,2020,3,0.19,True,4.2
143515,TXN_2020_00079961_DUP,2020-08-15,CUST_2015_00001186,PROD_000646,Apple iPhone 12 Pro 64GB White,Electronics,Smartphones,Apple,223856.56,0.00,...,False,NaN,5.0,Delivered,8,2020,3,0.19,True,4.2


In [54]:
duplicates.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().sort_values(ascending=False).head(10)

customer_id         product_id   order_date  original_price_inr
CUST_2015_00000020  PROD_001826  2020-11-20  26420.31              2
CUST_2015_00000046  PROD_000061  2020-11-15  102173.68             2
CUST_2015_00000122  PROD_000743  2020-05-03  42295.50              2
CUST_2015_00000134  PROD_000040  2020-10-28  184502.49             2
CUST_2015_00001186  PROD_000646  2020-08-15  223856.56             2
CUST_2015_00001668  PROD_000055  2020-11-13  67413.39              2
CUST_2015_00001991  PROD_000095  2020-01-27  28811.36              2
CUST_2015_00003283  PROD_001923  2020-03-31  64365.82              2
CUST_2015_00003558  PROD_001655  2020-07-20  83155.12              2
CUST_2015_00003636  PROD_000790  2020-05-02  42186.49              2
dtype: int64

In [55]:
dup_groups = df.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().reset_index(name="count")

dup_groups = dup_groups[dup_groups["count"] > 1]

In [56]:
dup_rows = df.merge(
    dup_groups,
    on=["customer_id","product_id","order_date","original_price_inr"],
    how="inner"
)

In [57]:
dup_rows[["customer_id","product_id","quantity","count"]].head()

,customer_id,product_id,quantity,count
0,CUST_2020_00031761,PROD_001614,2,2
1,CUST_2020_00043247,PROD_000786,2,2
2,CUST_2020_00046302,PROD_000678,1,2
3,CUST_2020_00000131,PROD_000189,1,2
4,CUST_2020_00044914,PROD_000314,1,2


In [58]:
df_clean = df.drop_duplicates(
    subset=["customer_id","product_id","order_date","original_price_inr"],
    keep="first"
)

In [59]:
df_clean.duplicated(
    subset=["customer_id","product_id","order_date","original_price_inr"]
).sum()

np.int64(0)

In [60]:
df["transaction_id"].duplicated().sum()

np.int64(0)

In [61]:
df = df.drop_duplicates(subset="transaction_id", keep="first")

In [62]:
df.duplicated(
    subset=["customer_id", "product_id", "order_date", "final_amount_inr"]
).sum()

np.int64(715)

In [63]:
df["transaction_id"].duplicated().sum()

np.int64(0)

Question 9
The dataset contains outlier prices where some products show prices 100x higher than expected due to data entry errors (decimal point issues). Identify and correct these outliers using statistical methods and domain knowledge.


In [64]:
# ── FIX 1: Negative prices ──────────────────────────
neg_mask = df["original_price_inr"] < 0
df.loc[neg_mask, "original_price_inr"] = df.loc[neg_mask, "original_price_inr"].abs()
print(f"Negative prices fixed: {neg_mask.sum()}")

# ── FIX 2: Outliers (100x decimal error) ────────────
subcategory_caps = {
    "Smart Watch":        60000,
    "Tablets":            110000,
    "Smartphones":        250000,
    "Laptops":            300000,
    "TV & Entertainment": 350000,
    "Audio":              80000,
}

outlier_mask = df.apply(
    lambda row: row["original_price_inr"] > subcategory_caps.get(row["subcategory"], 999999),
    axis=1
)
df.loc[outlier_mask, "original_price_inr"] = (
    df.loc[outlier_mask, "original_price_inr"] / 100
).round(2)
print(f"Outliers fixed: {outlier_mask.sum()}")

# ── FIX 3: Recalculate ───────────────────────────────
df["discounted_price_inr"] = (df["original_price_inr"] * (1 - df["discount_percent"] / 100)).round(2)
df["subtotal_inr"] = (df["discounted_price_inr"] * df["quantity"]).round(2)
df["final_amount_inr"] = df["subtotal_inr"]

# ── VERIFY ───────────────────────────────────────────
print(df.groupby("subcategory", observed=True)["original_price_inr"]
      .describe()[["min","max","mean","50%"]].round(2))
print(f"\nNaN in final_amount_inr:   {df['final_amount_inr'].isna().sum()}")
print(f"Negative prices remaining: {(df['original_price_inr'] < 0).sum()}")


Negative prices fixed: 351
Outliers fixed: 7289
                         min        max       mean        50%
subcategory                                                  
Audio                 819.87   59899.10   24939.39   28598.53
Laptops              4171.27  291658.57  124809.17  112638.20
Smart Watch           610.75   75409.42   26888.46   31895.68
Smartphones          2510.84  279027.44   74580.10   50930.79
TV & Entertainment  12735.01  326569.59  157061.69  148888.69
Tablets              1149.00  114903.91   49063.13   46964.98

NaN in final_amount_inr:   0
Negative prices remaining: 0


In [65]:
# Check the products exceeding caps
for sub, cap in subcategory_caps.items():
    over = df[(df["subcategory"] == sub) & 
                    (df["original_price_inr"] > cap)]
    if len(over) > 0:
        print(f"\n{sub} (cap ₹{cap:,}):")
        print(over[["product_name", "original_price_inr"]].drop_duplicates().head(5))


Smart Watch (cap ₹60,000):
              product_name  original_price_inr
19508  Garmin Watch Deluxe            68768.39
40821      Samsung Tracker            75409.42

Tablets (cap ₹110,000):
                    product_name  original_price_inr
25439  Lenovo iPad 8GB RAM Black           114903.91

Smartphones (cap ₹250,000):
                          product_name  original_price_inr
40435       Apple iPhone 5s 16GB Black           257052.60
56781  Apple iPhone 11 Pro 128GB Black           273980.58
94772   Apple iPhone XS Max 64GB White           279027.44


In [66]:
# Check if any legitimate products were over-corrected in cleaned files
for sub, cap in subcategory_caps.items():
    over = df[(df["subcategory"] == sub) & 
                    (df["original_price_inr"] > cap)]
    if len(over) > 0:
        print(f"\n{sub} (cap ₹{cap:,}): {len(over)} rows over")
        print(over[["product_name", "original_price_inr"]].drop_duplicates().head(5))
    else:
        print(f"\n{sub}: ✅ All within cap")


Smart Watch (cap ₹60,000): 2 rows over
              product_name  original_price_inr
19508  Garmin Watch Deluxe            68768.39
40821      Samsung Tracker            75409.42

Tablets (cap ₹110,000): 1 rows over
                    product_name  original_price_inr
25439  Lenovo iPad 8GB RAM Black           114903.91

Smartphones (cap ₹250,000): 3 rows over
                          product_name  original_price_inr
40435       Apple iPhone 5s 16GB Black           257052.60
56781  Apple iPhone 11 Pro 128GB Black           273980.58
94772   Apple iPhone XS Max 64GB White           279027.44

Laptops: ✅ All within cap

TV & Entertainment: ✅ All within cap

Audio: ✅ All within cap


Question 10
Payment methods contain inconsistent naming: 'UPI/PhonePe/GooglePay', 'Credit Card/CREDIT_CARD/CC', 
'Cash on Delivery/COD/C.O.D'. Standardize payment method categories and create a clean categorical hierarchy.


In [67]:
# ── Standardize payment methods ────────────────────
payment_standardize = {
    # UPI variants
    "UPI": "UPI", "PhonePe": "UPI", "GooglePay": "UPI", "Google Pay": "UPI",
    "UPI/PhonePe": "UPI", "UPI/GooglePay": "UPI",

    # Credit Card variants
    "Credit Card": "Credit Card", "CREDIT_CARD": "Credit Card", "CC": "Credit Card",

    # Debit Card variants
    "Debit Card": "Debit Card", "DEBIT_CARD": "Debit Card", "DC": "Debit Card",

    # COD variants
    "Cash on Delivery": "COD", "COD": "COD", "C.O.D": "COD",

    # Others
    "Wallet": "Wallet",
    "Net Banking": "Net Banking",
    "BNPL": "BNPL"
}

df["payment_method"] = df["payment_method"].map(payment_standardize).fillna(df["payment_method"])

# ── Create categorical hierarchy ───────────────────
payment_category = {
    "UPI":          "Digital Payment",
    "Wallet":       "Digital Payment",
    "Net Banking":  "Digital Payment",
    "Credit Card":  "Card Payment",
    "Debit Card":   "Card Payment",
    "BNPL":         "Pay Later",
    "COD":          "Cash on Delivery"
}

df["payment_category"] = df["payment_method"].map(payment_category).astype("category")

# ── Verify ─────────────────────────────────────────
print(df["payment_method"].value_counts())
print(f"\nNaN in payment_category: {df['payment_category'].isna().sum()}")

payment_method
UPI            43308
COD            35629
Credit Card    25832
Debit Card     24499
Net Banking    11528
Wallet          2919
Name: count, dtype: int64

NaN in payment_category: 0


Handling Nan - in customer age group

In [68]:
df["customer_age_group"].dtype

dtype('O')

In [69]:
df["customer_age_group"] = df["customer_age_group"].fillna("Unknown")

Checking for object columns

In [70]:
df.select_dtypes(include="object").columns

Index(['transaction_id', 'customer_id', 'product_id', 'product_name',
       'category', 'subcategory', 'brand', 'customer_city', 'customer_state',
       'customer_tier', 'customer_spending_tier', 'customer_age_group',
       'payment_method', 'delivery_type', 'festival_name', 'return_status'],
      dtype='object')

In [71]:
# ── Optimize memory: convert to category dtype ───────
cat_columns = ["category", "subcategory", "customer_tier", 
               "customer_spending_tier", "customer_age_group",
               "payment_method", "delivery_type", 
               "festival_name", "return_status"]

df[cat_columns] = df[cat_columns].astype("category")

# Verify
print(df[cat_columns].dtypes)
print(f"\nMemory usage after optimization:")
print(df.memory_usage(deep=True).sum() / 1024**2, "MB")

category                  category
subcategory               category
customer_tier             category
customer_spending_tier    category
customer_age_group        category
payment_method            category
delivery_type             category
festival_name             category
return_status             category
dtype: object

Memory usage after optimization:
77.12303352355957 MB


In [72]:
# NaN summary for all columns
nan_summary = df.isna().sum()
nan_summary = nan_summary[nan_summary > 0].sort_values(ascending=False)

print(f"Total columns with NaN: {len(nan_summary)}")
print(f"Total rows in dataset: {df.shape[0]}")
print(f"\nNaN counts and percentages:")
print(pd.DataFrame({
    "NaN Count": nan_summary,
    "Percentage": (nan_summary / df.shape[0] * 100).round(2)
}))

Total columns with NaN: 2
Total rows in dataset: 143715

NaN counts and percentages:
                 NaN Count  Percentage
festival_name        99915       69.52
customer_rating      43625       30.36


In [73]:
df.to_csv("data_cleaning_2020.csv", index=False)
print("File saved successfully!")

File saved successfully!
